# FINGUARD – Network Analysis

Graph construction, suspicious transaction relationships, centrality analysis, and community detection.



In [ ]:
#Step 1 — Import Graph Libraries
import networkx as nx


In [ ]:
#Step 2 — Extract Suspicious Transactions
fraud_df = df[df['isFraud'] == 1]
print(fraud_df.shape)


In [ ]:
sample_fraud = fraud_df.head(100)


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd

# Create graph
G = nx.DiGraph()
fraud_df = df[df["isFraud"] == 1].sample(50, random_state=42)
normal_df = df[df["isFraud"] == 0].sample(200, random_state=42)

df_sample = pd.concat([fraud_df, normal_df])


# Build network
for _, row in df_sample.iterrows():
    
    sender = row["nameOrig"]
    receiver = row["nameDest"]
    fraud = row["isFraud"]
    
    # Add nodes with fraud attribute
    G.add_node(sender, fraud=fraud)
    G.add_node(receiver, fraud=fraud)
    
    # Add transaction edge
    G.add_edge(sender, receiver)

# Separate nodes by fraud status
fraud_nodes = [n for n,d in G.nodes(data=True) if d["fraud"] == 1]
normal_nodes = [n for n,d in G.nodes(data=True) if d["fraud"] == 0]

# Graph layout
pos = nx.spring_layout(G, k=0.5, seed=42)

plt.figure(figsize=(12,10))

# Draw normal nodes
nx.draw_networkx_nodes(
    G,
    pos,
    nodelist=normal_nodes,
    node_color="#4C72B0",
    node_size=200,
    label="Normal Accounts"
)

# Draw fraud nodes
nx.draw_networkx_nodes(
    G,
    pos,
    nodelist=fraud_nodes,
    node_color="#DD3B3B",
    node_size=250,
    label="Fraud Accounts"
)

# Draw edges
nx.draw_networkx_edges(
    G,
    pos,
    edge_color="gray",
    alpha=0.4,
    arrows=False
)

plt.title("Fraud Transaction Network", fontsize=16)

plt.legend()
plt.axis("off")

plt.show()


In [ ]:
#Step 5 — Degree Centrality (Find Key Fraud Accounts)
degree_centrality = nx.degree_centrality(G)
top_accounts = sorted( 
    degree_centrality.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

print("Top Suspicious Accounts:")
print(top_accounts)


In [ ]:
#Step 6 — Betweenness Centrality (Money Flow Controllers)
betweenness = nx.betweenness_centrality(G)

top_betweenness = sorted(
    betweenness.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

print("Accounts controlling transaction flow:")
print(top_betweenness)


In [ ]:
#Step 7 — Community Detection (Fraud Rings)
from networkx.algorithms.community import greedy_modularity_communities

communities = greedy_modularity_communities(G)

print("Number of communities detected:", len(communities))


In [ ]:
# Step 8 — Visualize Communities
communities = list(communities)
node_colors = []
for node in G.nodes():
    for i, com in enumerate(communities):
        if node in com:
            node_colors.append(i)
            break
plt.figure(figsize=(10,8))
pos = nx.spring_layout(G)
nx.draw(
    G,  
    pos,
    node_color=node_colors,
    node_size=50,
    cmap=plt.cm.Set3,
    with_labels=False
)
plt.title("Fraud Communities in Transaction Network")
plt.show()


CENTALITY ANALYSIS
¶

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

# Calculate centrality
degree_centrality = nx.degree_centrality(G)
betweenness_centrality = nx.betweenness_centrality(G)
closeness_centrality = nx.closeness_centrality(G)

# Create dataframe
centrality_df = pd.DataFrame({
    "Account": list(degree_centrality.keys()),
    "Degree": list(degree_centrality.values()),
    "Betweenness": list(betweenness_centrality.values()),
    "Closeness": list(closeness_centrality.values())
})

# Sort by degree centrality
top_accounts = centrality_df.sort_values("Degree", ascending=False).head(10)

print("\nTop 10 Influential Accounts\n")
print(top_accounts.to_string(index=False))


In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# Calculate Degree Centrality
degree_centrality = nx.degree_centrality(G)

# Create DataFrame
centrality_df = pd.DataFrame({
    "Account": list(degree_centrality.keys()),
    "Degree Centrality": list(degree_centrality.values())
})

# Get Top 10 Accounts
top_accounts = centrality_df.sort_values(
    "Degree Centrality",
    ascending=False
).head(10)

print(top_accounts)

# Plot
plt.figure(figsize=(10,6))

plt.bar(
    top_accounts["Account"],
    top_accounts["Degree Centrality"]
)

plt.xticks(rotation=45)

plt.title("Top Influential Accounts in Transaction Network")
plt.xlabel("Account ID")
plt.ylabel("Degree Centrality")

plt.tight_layout()
plt.show()


GNN
¶

In [ ]:
# Install PyTorch Geometric (GNN library)
!pip install torch torchvision
!pip install torch_geometric
!pip install torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-2.0.0+cpu.html


In [ ]:
# Step 1 — Uninstall conflicting packages
!pip uninstall torch_sparse torch_scatter torch_geometric -y


In [ ]:
# Step 2 — Reinstall correctly
!pip install torch_geometric


In [ ]:
import torch
from torch_geometric.data import Data
print("PyG loaded successfully!")
